# Highest Price
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Binary Search, Hash Tables · **Difficulty/Frequency:** Very Common (8/10)


## Concepts

**What this problem is really testing:**
- Hash-map grouping, for instant lookups
- Sorting, to put related entries next to each other
- Precomputing a "running maximum so far" so range/checkpoint queries can use binary search

**Why each one shows up here:**
- The base question — "highest price at timestamp T" — is really "give me everything with this exact key". That's a hash map's specialty.
- The follow-up — "highest price at timestamp T, *as of* checkpoint K" — adds a second dimension: not just the key, but *when* you're asking. The trick: a **running maximum only ever goes up**. That's exactly the property that makes **binary search over a sorted/monotonic sequence** work.

**The one idea to hold onto:** precompute once, query many times. Whenever a problem says "answer lots of queries against the same fixed data", ask what you can build up front so each query becomes O(1) or O(log n) instead of a fresh scan.

---

### Quick primers — the building blocks used below

**What is a Hash Map?**
- A hash map (Python `dict`) stores key → value pairs by hashing the key to a slot, giving **O(1) average** insert/lookup/delete.
- **In Python:** use `dict`, or `collections.defaultdict(list)` when one key needs to map to a *growing* collection (here: every price seen at one timestamp).

**Sorting to make related things sit next to each other.**
- Sorting turns "all the things related to X" into one contiguous block — which is what unlocks range scans and binary search.
- **Cost:** O(n log n), paid once, up front.

**What is Binary Search?**
- Binary search finds a target (or where it would go) in a **sorted** sequence by repeatedly cutting the search range in half: check the middle, throw away the half that can't contain the answer, repeat.
- **Cost:** O(log n) per query — each comparison eliminates half of what's left.
- **In Python:** the `bisect` module. `bisect.bisect_right(a, x)` gives the insertion point just past any entries equal to `x` — subtract 1 and you get "the rightmost index whose key is ≤ x".

**Prefix-maximum (a running max that only ever goes up).**
- If you precompute, for every position `i`, "the max of everything from the start up to `i`", that sequence can only stay flat or increase — it's **monotonic** by construction.
- A monotonic sequence is exactly what makes binary search meaningful here: "the answer as of checkpoint k" becomes "the running-max value at the latest recorded point ≤ k" — found by binary-searching the recorded checkpoints.


## Problem Statement

Given a list of `(timestamp, price)` entries (timestamps **not** necessarily sorted, and possibly repeated), implement `highest_price_at_timestamp(entries, target_ts)` returning the max price recorded at `target_ts` (or `None`/no entry).

**Official follow-up -- checkpoints:** after each `(timestamp, price)` entry is processed, a checkpoint is created (checkpoint `k` = state after the k-th entry in original input order). Given a timestamp `t` and a checkpoint `k`, return the maximum price for `t` among entries `0..k`.

**Example**

```python
entries = [(1, 100), (2, 150), (1, 120), (3, 90), (2, 200)]
highest_price_at_timestamp(entries, 1)          # -> 120  (max of 100, 120)
max_price_at_checkpoint(entries, 2, checkpoint=1)   # -> 150 (only entries[0..1] considered)
max_price_at_checkpoint(entries, 2, checkpoint=4)   # -> 200 (entries[0..4], both ts=2 rows seen)
```


### Base question -- Approach 1 -- Naive (scan every query)

**Idea:** for each query, scan the whole list and track the max price among entries matching `target_ts`.

**Time complexity:** O(n) **per query** -- no precomputation, so repeated queries redo all the work every time.

**Space complexity:** O(1) extra.


In [ ]:
from typing import List, Optional, Tuple

Entry = Tuple[int, float]


def highest_price_naive(entries: List[Entry], target_ts: int) -> Optional[float]:
    best = None
    for ts, price in entries:                        # O(n) every single call
        if ts == target_ts:
            best = price if best is None else max(best, price)
    return best


### Base question -- Approach 2 -- Sort once, then scan a contiguous run

**Idea:** sort by timestamp so every entry for `target_ts` becomes contiguous, then scan (or binary-search the boundaries of) that run. Sorting is paid **once**; each query only touches its own run.

**Time complexity:** O(n log n) once to sort; each query is O(log n + m) if you binary-search the run's boundaries (m = entries at that timestamp), or O(n) worst case with a plain linear scan that stops early once timestamps exceed the target (as below -- simpler code, same asymptotic worst case, but never does wasted work past the target).

**Space complexity:** O(1) extra if sorting the list in place (or O(n) if you must preserve the caller's original order and sort a copy).


In [ ]:
def highest_price_sorted(entries: List[Entry], target_ts: int) -> Optional[float]:
    ordered = sorted(entries, key=lambda e: e[0])     # O(n log n), once per call in this signature
    best = None
    for ts, price in ordered:
        if ts == target_ts:
            best = price if best is None else max(best, price)
        elif ts > target_ts:
            break                                      # sorted -- nothing further can match
    return best


### Base question -- Approach 3 -- Optimal (hash map grouped by timestamp)

**Idea:** group all prices by timestamp once into a `dict[timestamp] -> list[price]` (or keep a running max per timestamp directly), then every query is a single O(1) lookup plus an O(m) max over that timestamp's own entries.

**Time complexity:** O(n) to build the map once; **O(1)** amortized per query if you track a running max while building (no per-query scan at all), or O(m) if you store the raw list and take `max()` at query time.

**Space complexity:** O(n) -- the map holds every entry exactly once.


In [ ]:
from collections import defaultdict
from typing import Dict


def build_price_index(entries: List[Entry]) -> Dict[int, float]:
    """Precompute timestamp -> running max price, once."""
    index: Dict[int, float] = {}
    for ts, price in entries:
        index[ts] = price if ts not in index else max(index[ts], price)
    return index


def highest_price_indexed(index: Dict[int, float], target_ts: int) -> Optional[float]:
    return index.get(target_ts)      # O(1) -- all the work already happened in build_price_index


### Follow-up -- checkpoint queries -- Approach 1 -- Naive (rescan the prefix)

**Idea:** a checkpoint is just "how much of the original, order-preserved list to consider". Rescanning `entries[0..checkpoint]` for every query is correct but repeats work across queries that share a timestamp or overlap in range.

**Time complexity:** O(k) per query, where k is the checkpoint index.

**Space complexity:** O(1) extra.


In [ ]:
def max_price_at_checkpoint_naive(entries: List[Entry], target_ts: int, checkpoint: int) -> Optional[float]:
    best = None
    for i in range(checkpoint + 1):                   # re-scans the same prefix on every call
        ts, price = entries[i]
        if ts == target_ts:
            best = price if best is None else max(best, price)
    return best


### Follow-up -- checkpoint queries -- Approach 2 -- Optimal (running max per timestamp + binary search)

**Idea:** process `entries` once, in original order. For each timestamp, keep a **running max** and append `(checkpoint_index, running_max_so_far)` to that timestamp's own list. Because the running max only ever increases (or stays flat) as the checkpoint advances, that per-timestamp list of `(checkpoint, running_max)` pairs is monotonic in checkpoint -- a query `(t, k)` becomes "binary search `t`'s list for the rightmost recorded checkpoint <= k", then read off its running max.

**Time complexity:** O(n) to preprocess; **O(log n)** per query (binary search within one timestamp's own list, whose length is at most n).

**Space complexity:** O(n) -- exactly one `(checkpoint, running_max)` entry is stored per original input row.


In [ ]:
import bisect


class PriceTracker:
    """Preprocesses once; answers (timestamp, checkpoint) queries in O(log n)."""

    def __init__(self, entries: List[Entry]) -> None:
        self.checkpoints: Dict[int, List[Tuple[int, float]]] = defaultdict(list)
        running: Dict[int, float] = {}
        for i, (ts, price) in enumerate(entries):
            running[ts] = price if ts not in running else max(running[ts], price)
            self.checkpoints[ts].append((i, running[ts]))   # monotonic in i by construction

    def query(self, target_ts: int, checkpoint: int) -> Optional[float]:
        if target_ts not in self.checkpoints:
            return None
        cps = self.checkpoints[target_ts]
        # bisect_right on (checkpoint, +inf) finds the insertion point just past any
        # (checkpoint, *) tuple -- subtracting 1 gives the rightmost entry with index <= checkpoint.
        idx = bisect.bisect_right(cps, (checkpoint, float("inf"))) - 1
        return cps[idx][1] if idx >= 0 else None


## Verification

Check every approach against the worked example, agreement between approaches on random-ish data, and the documented edge cases.

In [ ]:
entries = [(1, 100), (2, 150), (1, 120), (3, 90), (2, 200)]

# --- Base question: all three approaches must agree ---
assert highest_price_naive(entries, 1) == 120
assert highest_price_sorted(entries, 1) == 120
index = build_price_index(entries)
assert highest_price_indexed(index, 1) == 120

assert highest_price_naive(entries, 2) == 200
assert highest_price_sorted(entries, 2) == 200
assert highest_price_indexed(index, 2) == 200

# Timestamp with no entries -> None
assert highest_price_naive(entries, 99) is None
assert highest_price_sorted(entries, 99) is None
assert highest_price_indexed(index, 99) is None

# --- Checkpoint follow-up ---
# checkpoint=1 -> entries[0..1] = [(1,100),(2,150)]
assert max_price_at_checkpoint_naive(entries, 1, checkpoint=1) == 100
assert max_price_at_checkpoint_naive(entries, 2, checkpoint=1) == 150
assert max_price_at_checkpoint_naive(entries, 2, checkpoint=0) is None   # ts=2 hasn't appeared yet

tracker = PriceTracker(entries)
for ts in (1, 2, 3):
    for k in range(len(entries)):
        assert tracker.query(ts, k) == max_price_at_checkpoint_naive(entries, ts, k), (ts, k)
assert tracker.query(99, 4) is None                 # unknown timestamp
assert tracker.query(2, 4) == 200                   # full prefix -> both ts=2 rows seen

# --- Cross-check base vs. checkpoint at the final checkpoint (should agree) ---
last = len(entries) - 1
for ts in (1, 2, 3):
    assert tracker.query(ts, last) == highest_price_indexed(index, ts)

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **Entries arriving as an unbounded stream.** You can't store everything, but you don't need to: keep only a running max **per timestamp you might still be queried about**, and discard a timestamp's state once you can prove no future query will reference it (e.g. queries only ever ask about "recent" timestamps, or arrive in a bounded-lag order).
- **Supporting updates to a past timestamp's price.** `PriceTracker`'s checkpoint lists are append-only and assume prices never change retroactively -- an update would require recomputing every `running_max` recorded *after* that point for that timestamp. A **Fenwick tree (BIT)** or **segment tree** keyed by checkpoint index, storing max instead of sum, supports O(log n) point updates and O(log n) prefix-max queries, at the cost of more code than the simple monotonic-list trick.
- **Range queries over `[t1, t2]` instead of one timestamp.** Sort entries by timestamp and build a **sparse table** (O(n log n) preprocessing, O(1) range-max query, but immutable once built) or a **segment tree** (O(n) build, O(log n) query, supports updates) over the sorted price array.
- **Checkpoints defined by wall-clock arrival time instead of entry count.** Each entry would need its own arrival timestamp; a query becomes "among entries with arrival time <= T, what's the max price for commodity timestamp t" -- structurally the same problem with arrival-time replacing checkpoint-index as the monotonic query axis.
- **O(1) checkpoint queries with more preprocessing.** Possible with a persistent (versioned) structure or a full 2D prefix-max table, but both trade significant extra space (and, for the 2D table, O(n·distinct timestamps) preprocessing) for dropping O(log n) to O(1) -- rarely worth it unless queries vastly outnumber updates.


## Empirical complexity check

Compare the base question's **naive per-query scan** (Approach 1, O(n) every call) against the **precomputed hash index** (Approach 3, O(n) once + O(1) per call), running the *same* fixed number of queries (`Q = 200`) against a growing dataset of size n. Naive total work grows with n (each of the 200 queries costs O(n)); indexed total work is dominated by the one-time O(n) build and should barely move as n grows relative to it... to see the O(n) vs O(1)-per-query gap cleanly, we instead scale **the number of queries** against a fixed-size dataset.

| Growth when queries double | Implies (indexed approach) |
|---|---|
| ~1x | O(1) per query -- total time barely changes as query count grows, since the O(n) build is a one-time fixed cost |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

FIXED_N = 5000
FIXED_ENTRIES = [(i % 500, float(i)) for i in range(FIXED_N)]   # 500 distinct timestamps


def run_naive_queries(num_queries):
    for q in range(num_queries):
        highest_price_naive(FIXED_ENTRIES, q % 500)      # O(n) EVERY call


def run_indexed_queries(num_queries):
    index = build_price_index(FIXED_ENTRIES)              # O(n) ONCE per call to this function
    for q in range(num_queries):
        highest_price_indexed(index, q % 500)              # O(1) each


def make_worst_case(num_queries):
    return (num_queries,)


solutions = {
    "naive (O(n) per query)": run_naive_queries,
    "indexed (O(n) build + O(1)/query)": run_indexed_queries,
}
sizes = [200, 400, 800, 1600]     # this is now query COUNT, not dataset size
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Group by key once, query many times.** Any "find X matching key K" question repeated across many K values is a hash-map-grouping problem -- pay O(n) once, get O(1) (or O(m) for the group's own size) per query after.
- **Sorting creates contiguity, which unlocks range techniques.** Once related items are adjacent, a linear scan-with-early-break or a binary search over boundaries both become viable, whereas neither works on unsorted data.
- **Prefix aggregates turn "up to point K" queries into O(1)/O(log n) lookups.** A running max/sum/count array precomputed once answers any prefix query without re-scanning -- classic in problems phrased as "as of checkpoint/time/index K".
- **Monotonic sequence -> binary search is on the table.** Recognizing that a running maximum can never decrease is what licenses `bisect` here; the same recognition unlocks binary search in "smallest x such that condition(x) holds" problems generally.
- **State your policy for "no matching entry" explicitly.** `None` vs. raising vs. a sentinel value -- pick one and say why, especially once the interviewer starts asking about edge cases.
- **Related problems:** two-sum-style hash grouping, "first bad version" / monotonic binary search, prefix-sum range queries, LRU/LFU-style running-state tracking.
- **Common pitfalls:** forgetting that binary search requires the *searched* sequence (here, checkpoint indices) to be sorted/monotonic -- it's the checkpoint list per timestamp that's monotonic, not the raw input order; conflating "sorted by timestamp" with "sorted by checkpoint/arrival order" -- they're different axes and the two follow-up variants use different ones.
